In [ ]:
import os
os.environ["PYOPENGL_PLATFORM"] = "egl"

In [ ]:
import random
from collections import defaultdict

import trimesh
import numpy as np
import networkx as nx

class BurnSimulation:
    def __init__(self, obj_file_path):
        self.human_mesh, self.eyes_mesh, self.orbit_and_oral_mesh = self._load_obj(obj_file_path)
        self.face_graph = self._create_face_adjacency_graph(self.human_mesh)
        self.all_faces = np.arange(len(self.human_mesh.faces))
        self.face_areas = self.human_mesh.area_faces
        self.total_area = self.human_mesh.area
        self.burned_faces = set()

    @property
    def burned_area(self):
        return np.sum(self.face_areas[list(self.burned_faces)])

    @property
    def burn_ratio(self):
        return self.burned_area / self.total_area

    def _create_submesh(self, mesh, face_indices):
        if len(face_indices) == 0:
            return trimesh.Trimesh(process=False)

        selected_faces = mesh.faces[face_indices]
        unique_vertex_indices, inverse_map = np.unique(selected_faces, return_inverse=True)

        sub_vertices = mesh.vertices[unique_vertex_indices]
        sub_uvs = mesh.visual.uv[unique_vertex_indices]
        remapped_faces = inverse_map.reshape(selected_faces.shape)
        visual = trimesh.visual.texture.TextureVisuals(uv=sub_uvs, material=mesh.visual.material)
        return trimesh.Trimesh(vertices=sub_vertices, faces=remapped_faces, visual=visual, process=False)

    def _load_obj(self, obj_file_path):
        resolver = trimesh.resolvers.FilePathResolver(os.path.dirname(obj_file_path))
        loaded_data = trimesh.load(obj_file_path, force=None, resolver=resolver)
        meshes_to_process = []
        for _, geom in loaded_data.geometry.items():
            if isinstance(geom, trimesh.Trimesh):
                meshes_to_process.append(geom)
        sorted_meshes = sorted(meshes_to_process, key=lambda x:len(x.faces), reverse=True)
        human_mesh = sorted_meshes[0]
        eye_mesh = sorted_meshes[1]
        human_mesh.merge_vertices()
        eye_mesh.merge_vertices()
        human_mesh.visual.material.name = human_mesh.visual.material.name.split(".")[0]
        eye_mesh.visual.material.name = eye_mesh.visual.material.name.split(".")[0]

        non_main_body_face_indices = np.load("non_main_body_face_indices.npy")
        main_body_face_indices = np.setdiff1d(np.arange(len(human_mesh.faces)), non_main_body_face_indices)
        new_human_mesh = self._create_submesh(human_mesh, main_body_face_indices)
        orbit_and_oral_mesh = self._create_submesh(human_mesh, non_main_body_face_indices)

        inner_layer_eyes_face_indices = np.load("inner_layer_eyes_face_indices.npy")
        eye_mesh = self._create_submesh(eye_mesh, inner_layer_eyes_face_indices)

        return new_human_mesh, eye_mesh, orbit_and_oral_mesh

    def _create_face_adjacency_graph(self, mesh):
        vertices = mesh.vertices
        faces = mesh.faces
        coord_to_new_index = {}

        def get_or_add_vertex_index(vertex):
            rounded_coord = tuple(vertex)
            if rounded_coord not in coord_to_new_index:
                coord_to_new_index[rounded_coord] = len(coord_to_new_index)
            return coord_to_new_index[rounded_coord]

        faces_with_merged_vertex_ids = []
        for face in faces:
            merged_vertex_ids = [get_or_add_vertex_index(vertices[j]) for j in face]
            faces_with_merged_vertex_ids.append(merged_vertex_ids)

        merged_vertex_to_original_faces = defaultdict(set)
        for original_face_idx, merged_vertex_ids in enumerate(faces_with_merged_vertex_ids):
            for merged_vertex_id in merged_vertex_ids:
                merged_vertex_to_original_faces[merged_vertex_id].add(original_face_idx)

        G = nx.Graph()
        G.add_nodes_from(range(len(faces)))

        for original_face_set in merged_vertex_to_original_faces.values():
            original_face_list = list(original_face_set)
            for i in range(len(original_face_list)):
                for j in range(i + 1, len(original_face_list)):
                    G.add_edge(original_face_list[i], original_face_list[j])

        return G

    def _find_boundary_face(self, current_burn_set):
        all_processed = current_burn_set | self.burned_faces

        random_burn_set = list(current_burn_set)
        np.random.shuffle(random_burn_set)
        for face in random_burn_set:
            for neighbor in self.face_graph.neighbors(face):
                if neighbor not in all_processed:
                    return face
        return None

    def _grow_burn_faces(self, target_face_area, current_burn_set):
        available_faces = np.setdiff1d(self.all_faces, list(self.burned_faces|current_burn_set))
        if len(available_faces) == 0:
            return set()

        iter_face_to_burn = set()
        iter_face_area = 0.0

        if not current_burn_set: # new generation, start from a random face
            start_face = random.choices(available_faces, weights=self.face_areas[available_faces], k=1)[0]
        else: # existing generation, start from the boundary of current burn set
            boundary_face = self._find_boundary_face(current_burn_set)
            if boundary_face:
                start_face = boundary_face
            else: # all current burn faces are surrounded by already burned faces
                return set()

        queue = [start_face]
        bfs_history = set()

        while queue and iter_face_area < target_face_area * self.total_area:
            current_face = queue.pop(0)
            if current_face in iter_face_to_burn or current_face in bfs_history:
                continue

            bfs_history.add(current_face)

            for neighbor_face in self.face_graph.neighbors(current_face):
                if neighbor_face not in iter_face_to_burn and neighbor_face not in bfs_history:
                    queue.append(neighbor_face)

            if current_face not in current_burn_set:
                iter_face_to_burn.add(current_face)
                iter_face_area += self.face_areas[current_face]
        return iter_face_to_burn

    def apply_burn_effect(self, burn_percentage=0.1):
        if burn_percentage <= 0:
            return

        available_faces = np.setdiff1d(self.all_faces, list(self.burned_faces))
        if len(available_faces) == 0:
            return

        target_burn_area = self.total_area * burn_percentage
        available_total_area = np.sum(self.face_areas[available_faces])

        if available_total_area < target_burn_area:
            face_to_burn = available_faces
        else:
            face_to_burn = set()
            current_burn_area = 0.0
            remaining_area = target_burn_area

            while remaining_area > 0:
                max_possible_new_area = min(remaining_area, available_total_area - current_burn_area)
                if max_possible_new_area <= 0:
                    break

                random_ratio = np.random.uniform(0.001, min(0.005, remaining_area))
                target_patch_area = random_ratio * self.total_area

                new_area = self._grow_burn_faces(target_patch_area, face_to_burn)
                if len(new_area) == 0:
                    break

                face_to_burn.update(new_area)
                new_burn_area = np.sum(self.face_areas[list(new_area)])
                current_burn_area += new_burn_area
                remaining_area -= new_burn_area

        face_to_burn = list(face_to_burn)
        self.burned_faces.update(face_to_burn)
        real_burn_ratio = np.sum(self.face_areas[list(face_to_burn)]) / self.total_area
        if real_burn_ratio<burn_percentage:
            self.apply_burn_effect(burn_percentage - real_burn_ratio)

    def reset(self):
        self.burned_faces.clear()

    def export(self):
        all_faces_indices = np.arange(len(self.human_mesh.faces))
        burned_face_indices = np.array(list(self.burned_faces))
        unburned_face_indices = np.setdiff1d(all_faces_indices, burned_face_indices)

        burned_mesh = self._create_submesh(self.human_mesh, burned_face_indices)
        unburned_mesh = self._create_submesh(self.human_mesh, unburned_face_indices)

        return burned_mesh, unburned_mesh



In [ ]:
from PIL import Image

NOISE = Image.open("noise.png").convert("RGB")

def random_noise(min_size=0.25, max_size=2, output_size=1024):
    random_size = int(random.uniform(min_size, max_size)*output_size)
    l = random.randint(0, NOISE.width - random_size)
    t = random.randint(0, NOISE.height - random_size)
    r = l + random_size
    b = t + random_size
    noise_cropped = NOISE.crop((l, t, r, b)).resize((output_size, output_size), resample=Image.BILINEAR)
    return np.array(noise_cropped)

In [ ]:
import random

import cv2
import pyrender
import numpy as np

class Renderer:
    def __init__(self, burn_simulation):
        self.bbox_min, self.bbox_max = burn_simulation.human_mesh.bounds
        self.burned_human_mesh,self.unburned_human_mesh = burn_simulation.export()
        self.eyes_mesh = burn_simulation.eyes_mesh
        self.orbit_and_oral_mesh = burn_simulation.orbit_and_oral_mesh
        self.light = pyrender.DirectionalLight(intensity=25)
        self.camera, self.camera_pose = self._get_camera()

    def _get_camera(self, front_view=True, fov_deg=24, angle_x=0, angle_y=0):
        center = (self.bbox_min + self.bbox_max) / 2.0

        bbox_diagonal = np.linalg.norm(self.bbox_max - self.bbox_min)
        radius = bbox_diagonal / 2.0

        fov_rad = fov_deg * (np.pi / 180)
        camera = pyrender.PerspectiveCamera(yfov=fov_rad)

        min_distance = radius / np.tan(fov_rad / 2)
        distance = min_distance * 1.05

        pitch_rad = np.radians(angle_x)
        yaw_rad = np.radians(angle_y)

        dx = np.sin(yaw_rad)
        dy = np.sin(pitch_rad)
        dz = np.cos(pitch_rad) * np.cos(yaw_rad)

        direction = np.array([dx, dy, dz])

        if not front_view:
            direction[2] *= -1

        direction = direction / np.linalg.norm(direction)

        eye = center + direction * distance

        target = center
        world_up = np.array([0.0, 1.0, 0.0])

        z_axis = eye - target
        z_axis /= np.linalg.norm(z_axis)

        x_axis = np.cross(world_up, z_axis)

        if np.linalg.norm(x_axis) < 1e-6:
            x_axis = np.cross(np.array([1.0, 0.0, 0.0]), z_axis)

        x_axis /= np.linalg.norm(x_axis)

        y_axis = np.cross(z_axis, x_axis)
        y_axis /= np.linalg.norm(y_axis)

        camera_pose = np.eye(4)
        camera_pose[:3, 0] = x_axis
        camera_pose[:3, 1] = y_axis
        camera_pose[:3, 2] = z_axis
        camera_pose[:3, 3] = eye

        return camera, camera_pose

    def random_camera_pose_and_light_intensity(self, front_view=True, fov_range=(24,28), angle_x_range=(-15,15), angle_y_range=(-15,15), intensity_range=(20,30)):
        self.camera, self.camera_pose = self._get_camera(front_view, random.uniform(*fov_range), random.uniform(*angle_x_range), random.uniform(*angle_y_range))
        self.light.intensity = random.uniform(*intensity_range)

    def _basic_render(self, burned_only=False, size=1024):
        scene = pyrender.Scene()

        scene.add(pyrender.Mesh.from_trimesh(self.burned_human_mesh))
        if not burned_only:
            scene.add(pyrender.Mesh.from_trimesh(self.unburned_human_mesh))
            scene.add(pyrender.Mesh.from_trimesh(self.eyes_mesh))
            scene.add(pyrender.Mesh.from_trimesh(self.orbit_and_oral_mesh))
        else:
            transparent_material = pyrender.MetallicRoughnessMaterial(
                metallicFactor=0.0,
                roughnessFactor=1.0,
                alphaMode="OPAQUE",
                baseColorFactor=[255.0, 255.0, 255.0, 255.0]
            )
            scene.add(pyrender.Mesh.from_trimesh(self.unburned_human_mesh, material=transparent_material))
            scene.add(pyrender.Mesh.from_trimesh(self.eyes_mesh, material=transparent_material))
            scene.add(pyrender.Mesh.from_trimesh(self.orbit_and_oral_mesh, material=transparent_material))

        scene.add(self.camera, pose=self.camera_pose)
        scene.add(self.light, pose=self.camera_pose)

        renderer = pyrender.OffscreenRenderer(viewport_width=size, viewport_height=size)
        color, depth = renderer.render(scene)
        renderer.delete()
        return color, depth

    def _blend_with_texture_and_fade_border(self, image, mask, texture, kernel_size=5, alpha_multiplier=0.5):
        image = image.astype(np.float32) / 255.0
        mask = mask.astype(np.float32)
        texture = texture.astype(np.float32) / 255.0

        kernel = np.ones((kernel_size, kernel_size), np.uint8)
        eroded_mask = cv2.erode(mask, kernel, iterations=1)
        border = mask - eroded_mask

        final_image = image.copy()

        multiplied_region = image * texture
        masked_multiplied = image + mask[:, :, None] * (multiplied_region - image)
        faded_blend = image + border[:, :, None] * alpha_multiplier * (multiplied_region - image)

        final_image = final_image * (1 - eroded_mask[:, :, None]) + masked_multiplied * eroded_mask[:, :, None]
        final_image = final_image * (1 - border[:, :, None]) + faded_blend * border[:, :, None]

        return np.clip(final_image * 255, 0, 255).astype(np.uint8)

    def render(self, render_size=1024, output_size=512):
        full_color, full_depth = self._basic_render(burned_only=False)
        full_mask = full_depth != 0
        burned_color, _ = self._basic_render(burned_only=True, size=render_size)
        burned_mask = np.logical_not(np.all(np.array(burned_color) == [255, 255, 255], axis=-1))
        noise = random_noise(output_size=render_size)

        rows = np.any(full_mask, axis=1)
        cols = np.any(full_mask, axis=0)
        rmin, rmax = np.where(rows)[0][[0, -1]]
        cmin, cmax = np.where(cols)[0][[0, -1]]

        full_color = full_color[rmin:rmax+1, cmin:cmax+1]
        full_mask = full_mask[rmin:rmax+1, cmin:cmax+1]
        burned_mask = burned_mask[rmin:rmax+1, cmin:cmax+1]
        noise = noise[rmin:rmax+1, cmin:cmax+1]

        blended_image = Image.fromarray(self._blend_with_texture_and_fade_border(full_color, burned_mask, noise))
        full_mask_image = Image.fromarray(full_mask)
        burned_mask_image = Image.fromarray(burned_mask)

        original_width, original_height = blended_image.size
        ratio = min(output_size / original_width, output_size / original_height)
        new_size = (int(original_width * ratio), int(original_height * ratio))

        blended_image = blended_image.resize(new_size)
        full_mask_image = full_mask_image.resize(new_size)
        burned_mask_image = burned_mask_image.resize(new_size)


        return blended_image, full_mask_image, burned_mask_image


In [ ]:
import torch
from torch.utils.data import Dataset

class MassHumanBurnBuilder(Dataset):
    burn_levels = 8
    views = 4
    def __init__(self, input_path="mass_humans", output_path="mass_human_burns"):
        self.input_path = input_path
        self.output_path = output_path
        os.makedirs(os.path.join(self.output_path,"images","front"), exist_ok=True)
        os.makedirs(os.path.join(self.output_path,"images","back"), exist_ok=True)
        os.makedirs(os.path.join(self.output_path,"full_masks","front"), exist_ok=True)
        os.makedirs(os.path.join(self.output_path,"full_masks","back"), exist_ok=True)
        os.makedirs(os.path.join(self.output_path,"burn_masks","front"), exist_ok=True)
        os.makedirs(os.path.join(self.output_path,"burn_masks","back"), exist_ok=True)

    def __len__(self):
        return len([file for file in os.listdir(self.input_path) if file.endswith(".obj")])

    def _generate_one_pair(self, burn_simulation,idx,level,view):
        renderer = Renderer(burn_simulation)
        renderer.random_camera_pose_and_light_intensity()
        image,full_mask,burn_mask = renderer.render()
        image.save(os.path.join(self.output_path,"images","front", f"{idx}_{level}_{view}.png"))
        full_mask.save(os.path.join(self.output_path,"full_masks","front", f"{idx}_{level}_{view}.png"))
        burn_mask.save(os.path.join(self.output_path,"burn_masks","front", f"{idx}_{level}_{view}.png"))

        renderer.random_camera_pose_and_light_intensity(False)
        image,full_mask,burn_mask = renderer.render()
        image.save(os.path.join(self.output_path,"images","back", f"{idx}_{level}_{view}.png"))
        full_mask.save(os.path.join(self.output_path,"full_masks","back", f"{idx}_{level}_{view}.png"))
        burn_mask.save(os.path.join(self.output_path,"burn_masks","back", f"{idx}_{level}_{view}.png"))


    def _generate_one_burn(self, burn_simulation, target_burn_ratio,idx,level):
        burn_times = random.randint(1,10)
        burn_points = sorted(random.uniform(0, target_burn_ratio) for _ in range(burn_times - 1))+[target_burn_ratio]

        for burn_point in burn_points:
            burn_simulation.apply_burn_effect(burn_point - burn_simulation.burn_ratio)

        for view in range(self.views):
            self._generate_one_pair(burn_simulation, idx, level, view)
        return burn_simulation.burn_ratio

    def __getitem__(self, idx):
        burn_simulation = BurnSimulation(os.path.join(self.input_path, f"human_{idx}.obj"))
        result = []
        burn_ratio = random.uniform(0.0, 0.05)
        real_burn_ratios = self._generate_one_burn(burn_simulation, burn_ratio, idx, 0)
        result.append(torch.tensor(real_burn_ratios))

        burn_ratio = random.uniform(0.05, 0.1)
        real_burn_ratios = self._generate_one_burn(burn_simulation, burn_ratio, idx, 1)
        result.append(torch.tensor(real_burn_ratios))

        burn_ratio = random.uniform(0.1, 0.2)
        real_burn_ratios = self._generate_one_burn(burn_simulation, burn_ratio, idx, 2)
        result.append(torch.tensor(real_burn_ratios))

        burn_ratio = random.uniform(0.2, 0.3)
        real_burn_ratios = self._generate_one_burn(burn_simulation, burn_ratio, idx, 3)
        result.append(torch.tensor(real_burn_ratios))

        burn_ratio = random.uniform(0.3, 0.4)
        real_burn_ratios = self._generate_one_burn(burn_simulation, burn_ratio, idx, 4)
        result.append(torch.tensor(real_burn_ratios))

        burn_ratio = random.uniform(0.4, 0.5)
        real_burn_ratios = self._generate_one_burn(burn_simulation, burn_ratio, idx, 5)
        result.append(torch.tensor(real_burn_ratios))

        burn_ratio = random.uniform(0.5, 0.7)
        real_burn_ratios = self._generate_one_burn(burn_simulation, burn_ratio, idx, 6)
        result.append(torch.tensor(real_burn_ratios))

        burn_ratio = random.uniform(0.7, 1.0)
        real_burn_ratios = self._generate_one_burn(burn_simulation, burn_ratio, idx, 7)
        result.append(torch.tensor(real_burn_ratios))
        
        return torch.stack(result), torch.tensor(idx)

In [ ]:
from torch.utils.data import DataLoader

dataset_builder=MassHumanBurnBuilder()
dataloader = DataLoader(dataset_builder, num_workers=32, shuffle=True, in_order=False)

In [ ]:
from tqdm import tqdm

burn_ratios = {}
for label,idx in tqdm(dataloader):
    burn_ratios[idx.item()] = label[0]

np.save("mass_human_burns/labels.npy", np.array([burn_ratios[i] for i in sorted(burn_ratios.keys())]))